In [6]:
import os
import librosa
import numpy as np
import pandas as pd
import msaf

In [7]:
def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=None)
    
    features = {}
    
    features["duration"] = librosa.get_duration(y=y, sr=sr)
    
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features["tempo"] = float(np.mean(tempo))
    
    features["rms_mean"] = np.mean(librosa.feature.rms(y=y))
    features["spectral_centroid_mean"] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    features["spectral_bandwidth_mean"] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    features["spectral_rolloff_mean"] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    features["zcr_mean"] = np.mean(librosa.feature.zero_crossing_rate(y))
    
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features["chroma_mean"] = np.mean(chroma)
    features["chroma_std"] = np.std(chroma)
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])
    
    return features

In [8]:
def structural_features(filepath):
    boundaries, labels = msaf.process(filepath, boundaries_id="foote")
    
    segment_lengths = np.diff(boundaries)
    labels_array = np.array(labels)
    non_negative_labels = labels_array[labels_array != -1]
    unique_labels = set(labels_array)
    
    entropy = 0
    for l in unique_labels:
        if l != -1:
            p = list(labels_array).count(l) / len(labels_array)
            entropy += -p * np.log2(p)
    
    return {
        "num_segments": len(segment_lengths),
        "avg_segment_length": np.mean(segment_lengths),
        "segment_length_std": np.std(segment_lengths),
        "num_structured_sections": len(non_negative_labels),
        "structure_ratio": len(non_negative_labels) / len(labels_array),
        "unique_sections": len(unique_labels) - (1 if -1 in unique_labels else 0),
        "label_entropy": entropy
    }

In [9]:
song_path = "song_mp3/2010/Imagine Dragons - Radioactive (Lyric Video).mp3"

audio_feats = extract_features(song_path)
struct_feats = structural_features(song_path)

combined = {**audio_feats, **struct_feats}
combined["song"] = os.path.basename(song_path)
combined["decade"] = "2010s"

df_example = pd.DataFrame([combined])
df_example

,duration,tempo,rms_mean,spectral_centroid_mean,spectral_bandwidth_mean,spectral_rolloff_mean,zcr_mean,chroma_mean,chroma_std,mfcc_1_mean,...,mfcc_13_std,num_segments,avg_segment_length,segment_length_std,num_structured_sections,structure_ratio,unique_sections,label_entropy,song,decade
0,186.084717,135.999178,0.321914,2028.466029,2460.357645,4040.700394,0.045641,0.435655,0.295004,-87.061829,...,8.459212,8,23.26059,10.096688,1,0.125,1,0.375,Imagine Dragons - Radioactive (Lyric Video).mp3,2010s
